# Analysing Stock Prices

In this notebook, we will be analysing stock prices using Python. We will be using the pandas library to read and manipulate the data, and the matplotlib library to visualize the data. To start of, we will import the `yfinance` libraries and read in the stock price data between `2007-1-1` to `2026-01-01`, and save them in the `prices` folder in the working directory.

For those viewing the project on GitHub, I have provided a download script within this notebook file which will fetch the stock price data from Yahoo Finance and save it in a `price` folder in your working directory .

In [1]:
%%writefile capture_stock_prices.py
import argparse
import csv
import os
import sys
import time
import yfinance as yf

START_DATE = "2007-01-01"
END_DATE = "2026-01-01"
PRICES_DIR = "prices"
SYMBOLS_FILE = "nasdaqlisted.txt"
REQUEST_DELAY = 0.05
MAX_RETRIES = 3

COLUMNS = ["date", "open", "high", "low", "close", "adj_close", "volume"]


def load_symbols(path, include_etfs=False):
    """Load ticker symbols from either a plain list or a NASDAQ listing file.

    Accepts both:
      - one ticker per line (blank lines and # comments ignored)
      - NASDAQ's pipe-delimited nasdaqlisted.txt

    For the NASDAQ format, test issues are always dropped and ETFs are
    dropped unless include_etfs is True.
    """
    if not os.path.exists(path):
        sys.exit(
            f"No symbol list found at '{path}'.\n"
            "Use a plain list, NASDAQ's nasdaqlisted.txt, or --symbols AAPL MSFT ..."
        )

    with open(path, encoding="utf-8") as f:
        first = f.readline()
        f.seek(0)

        # NASDAQ listing files are pipe-delimited with a Symbol column
        if "|" in first and "Symbol" in first:
            reader = csv.DictReader(f, delimiter="|")
            symbols, dropped_test, dropped_etf = [], 0, 0

            for row in reader:
                symbol = (row.get("Symbol") or "").strip().upper()

                # Trailing footer line: "File Creation Time: ...|||||||"
                if not symbol or symbol.startswith("FILE CREATION"):
                    continue
                # Test tickers exist purely for exchange system checks
                if row.get("Test Issue", "").strip().upper() == "Y":
                    dropped_test += 1
                    continue
                if not include_etfs and row.get("ETF", "").strip().upper() == "Y":
                    dropped_etf += 1
                    continue
                # Suffixed symbols (warrants, preferred, classes) use different
                # conventions on Yahoo and mostly 404 — skip them
                if not symbol.isalpha():
                    continue

                symbols.append(symbol)

            print(
                f"Parsed NASDAQ listing: {len(symbols)} symbols "
                f"({dropped_test} test issues, {dropped_etf} ETFs excluded)"
            )
            return symbols

        # Plain one-per-line list
        return [
            line.strip().upper()
            for line in f
            if line.strip() and not line.startswith("#")
        ]

def fetch_symbol(symbol, start, end):
    """Fetch one symbol's history. Returns a DataFrame, or None if unavailable."""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            df = yf.Ticker(symbol).history(
                start=start,
                end=end,
                interval="1d",
                auto_adjust=False,
            )
            if df.empty:
                return None
            return df
        except Exception as exc:
            if attempt == MAX_RETRIES:
                print(f"  {symbol}: failed after {MAX_RETRIES} attempts ({exc})")
                return None
            # Back off progressively rather than hammering a rate limit
            time.sleep(2 ** attempt)
    return None

def write_csv(symbol, df, out_dir):
    """Write the DataFrame to prices/<symbol>.csv in the target schema."""
    path = os.path.join(out_dir, f"{symbol}.csv")
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(COLUMNS)
        for date, row in df.iterrows():
            writer.writerow([
                date.strftime("%Y-%m-%d"),
                round(row["Open"], 4),
                round(row["High"], 4),
                round(row["Low"], 4),
                round(row["Close"], 4),
                round(row.get("Adj Close", row["Close"]), 4),
                int(row["Volume"]),
            ])
    return path

def main():
    parser = argparse.ArgumentParser(description="Download NASDAQ daily prices.")
    parser.add_argument("--symbols", nargs="+", help="Tickers to fetch (overrides symbols.txt)")
    parser.add_argument("--start", default=START_DATE, help="Start date, YYYY-MM-DD")
    parser.add_argument("--end", default=END_DATE, help="End date, YYYY-MM-DD")
    parser.add_argument("--out", default=PRICES_DIR, help="Output directory")
    parser.add_argument("--force", action="store_true", help="Re-download existing files")
    parser.add_argument("--symbol-file", default=SYMBOLS_FILE, help="Plain list or nasdaqlisted.txt")
    parser.add_argument("--include-etfs", action="store_true", help="Keep ETFs from a NASDAQ listing")
    parser.add_argument("--limit", type=int, help="Only fetch the first N symbols")
    args = parser.parse_args()

    symbols = args.symbols or load_symbols(args.symbol_file, args.include_etfs)
    symbols = [s.upper() for s in symbols]
    if args.limit:
        symbols = symbols[:args.limit]
    os.makedirs(args.out, exist_ok=True)

    print(f"{len(symbols)} symbols | {args.start} to {args.end} | -> {args.out}/\n")

    downloaded = skipped = failed = 0

    for i, symbol in enumerate(symbols, 1):
        out_path = os.path.join(args.out, f"{symbol}.csv")

        if os.path.exists(out_path) and not args.force:
            skipped += 1
            continue

        print(f"[{i}/{len(symbols)}] {symbol}", end=" ")
        df = fetch_symbol(symbol, args.start, args.end)

        if df is None:
            print("- no data")
            failed += 1
        else:
            write_csv(symbol, df, args.out)
            print(f"- {len(df)} rows")
            downloaded += 1

        time.sleep(REQUEST_DELAY)

    print(
        f"\nDone. {downloaded} downloaded, {skipped} already present, {failed} failed."
    )
    if failed:
        print("Failures are usually delisted tickers or symbols renamed since 2026.")

if __name__ == "__main__":
    main()

Overwriting capture_stock_prices.py


In [2]:
%run capture_stock_prices.py

Parsed NASDAQ listing: 2861 symbols (9 test issues, 301 ETFs excluded)
2861 symbols | 2007-01-01 to 2026-01-01 | -> prices/

[1/2861] AAAP 

$AAAP: Data doesn't exist for startDate = 1167627600, endDate = 1767243600
$AAPC: possibly delisted; no price data found  (1d 2007-01-01 -> 2026-01-01)


- no data
[6/2861] AAPC - no data
[8/2861] AAWW 

$AAWW: possibly delisted; no timezone found


- no data
[9/2861] AAXN 

...

- no data
[2859/2861] ZSAN 

$ZSAN: possibly delisted; no timezone found


- no data
[2861/2861] ZYNE 

$ZYNE: possibly delisted; no timezone found


- no data

Done. 0 downloaded, 1473 already present, 1388 failed.
Failures are usually delisted tickers or symbols renamed since 2026.


We were able to load the data for 1,427 symbols, which is more than sufficient for the purpose of this demonstration. If we were to use this in a commercial environment, we would need to consider additional factors such as data quality, data storage, and data processing requirements, and investigate why nearly half of our tickers failed. It is more than likely the ticket source file is outdated and carrying old ticker names.

I will also experiment with loading the files using various methods to see which would be more efficient for future purposes.

In [3]:
# Default Loading Method
import os, time
import pandas as pd

stock_prices = {}
start = time.time()
for fn in os.listdir("prices"):
    name = fn.split(".")[0]
    stock_prices[name] = pd.read_csv(os.path.join("prices", fn))
time_taken = time.time() - start
print(f"Loaded {len(stock_prices):,} tickers in {time_taken:.2f} seconds")

Loaded 1,473 tickers in 5.96 seconds


In [4]:
%%writefile file_ingest.py
# Initial read of all stock prices
import os
import pandas as pd

def mapper(file_chunk, directory):
    stock_prices = {}
    for file in file_chunk:
        name = os.path.splitext(file)[0]
        stock_prices[name] = pd.read_csv(os.path.join(directory, file), encoding="UTF-8")
    return stock_prices, len(file_chunk)

def reducer(a, b):
    merged = dict(a)
    merged.update(b)
    return merged

Overwriting file_ingest.py


In [7]:
# Using a MapReduce Function (with Parallelism)
import os, math, functools, importlib, time
from multiprocessing import Pool
from IPython.display import clear_output

import file_ingest
importlib.reload(file_ingest)
from file_ingest import mapper, reducer

def make_chunks(data, num_chunks):
    chunk_size = math.ceil(len(data) / num_chunks)
    return [data[i:i+chunk_size] for i in range(0, len(data), chunk_size)]

def map_reduce(data, directory, num_processes, mapper, reducer, num_chunks=20, report_every=100):
    chunks = make_chunks(data, num_chunks)
    worker = functools.partial(mapper, directory=directory)
    results = {}
    processed = 0
    next_report = report_every

    with Pool(num_processes) as pool:
        for chunk_prices, n_files in pool.imap(worker, chunks):
            results = reducer(results, chunk_prices)
            processed += n_files
            while processed >= next_report:
                print(f"  {next_report:,} files", flush=True)
                next_report += report_every

    return results

if __name__ == "__main__":
    directory = "prices"
    files = [f for f in os.listdir(directory) if f.endswith(".csv")]
    workers = max(1, os.cpu_count() // 2)
    start = time.time()
    stock_prices = map_reduce(files, directory, workers, mapper, reducer)
    time_taken = time.time() - start
    clear_output(wait=True)
    print(f"Loaded {len(stock_prices):,} tickers in {time_taken:.2f} seconds")

Loaded 1,473 tickers in 2.68 seconds


In [10]:
# Using a MapReduce Method (with Parquets)
import os, time
import pandas as pd
import pyarrow as pa
import pyarrow.csv as pv
from IPython.display import clear_output

directory = "prices"
schema = {"date": pa.date32(),
          "close": pa.float32(),
          "open": pa.float32(),
          "high": pa.float32(),
          "low": pa.float32(),
          "volume": pa.int64()
}
conversion = pv.ConvertOptions(column_types=schema, include_columns=list(schema))

files = sorted(f for f in os.listdir(directory) if f.lower().endswith(".csv"))
frames, failed, start = [], [], time.perf_counter()

for i, file in enumerate(files, 1):
    df = pv.read_csv(os.path.join(directory, file), convert_options=conversion).to_pandas()
    df["ticker"] = os.path.splitext(file)[0].lower()
    frames.append(df)
    if i % 1000 == 0:
        print(f"  {i:,}/{len(files):,}", flush=True)

combined = pd.concat(frames, ignore_index=True)
combined["ticker"] = combined["ticker"].astype("category")
stock_prices = {t: g.drop(columns="ticker").set_index("date") for t, g in combined.groupby("ticker", observed=True)}
clear_output(wait=True)

print(f"Loaded {len(stock_prices):,} tickers in {time.perf_counter() - start:.2f}s")
if failed:
    print(f"{len(failed)} failed:", *[f"  {f}: {e}" for f, e in failed[:10]], sep="\n")

Loaded 1,473 tickers in 4.71s


Based on the performance data, we can see that the MapReduce Function (with Parallelism) is significantly faster than the other methods.

This is unexpected, and we would normally expect the final method to be the fastest. This is because Parquet files are columnar storage format that allows for efficient data compression and faster data access. Additionally, using PyArrow's CSV reader and converter options allows for faster data ingestion and conversion to Pandas DataFrames.

Whilst unexpected, this may be due to the limited number of files being processed. It is possible that the overhead of reading and converting the files is greater than the benefits of using Parquet files and PyArrow's CSV reader and converter options for smaller datasets.

Below, I have submitted my code above to Claude for review, and it suggested further optimisations and combining both approaches. This further saves time, but this could be due to the use of ThreadPools to reduce the overhead of spawning processes.

In [11]:
import os, time
import pandas as pd
import pyarrow as pa
import pyarrow.csv as pv
from multiprocessing.dummy import Pool as ThreadPool   # threads, not processes

DIRECTORY = "prices"
WORKERS = os.cpu_count()

SCHEMA = {"date": pa.date32(), "close": pa.float32(), "open": pa.float32(),
          "high": pa.float32(), "low": pa.float32(), "volume": pa.int64()}
OPTS = pv.ConvertOptions(column_types=SCHEMA, include_columns=list(SCHEMA))
RO = pv.ReadOptions(use_threads=False)

def read_one(file):
    table = pv.read_csv(os.path.join(DIRECTORY, file), read_options=RO, convert_options=OPTS)
    return os.path.splitext(file)[0].lower(), table.to_pandas().set_index("date")

files = sorted(f for f in os.listdir(DIRECTORY) if f.lower().endswith(".csv"))
start = time.perf_counter()

with ThreadPool(WORKERS) as pool:
    stock_prices = dict(pool.map(read_one, files, chunksize=25))

print(f"Loaded {len(stock_prices):,} tickers in {time.perf_counter() - start:.2f}s")

Loaded 1,473 tickers in 2.10s
